# Capítulo 13. Agrupamiento k-means

**Aprendizaje y Clasificación Automática con R**  
**Autor:** Jesús Gilberto Rodríguez Escobedo

Este cuaderno es **independiente y autónomo**: puede abrirse directamente sin ejecutar capítulos anteriores.

1. Ejecute primero la celda **Preparación automática y autónoma del capítulo**.
2. Después ejecute las celdas en orden.
3. Si Colab reinicia la sesión, vuelva a ejecutar desde la primera celda.

[Volver al índice de cuadernos Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/00-indice-colabs.ipynb)


In [ ]:
# Preparación automática y autónoma del capítulo
options(repos = c(CRAN = "https://cloud.r-project.org"))

paquetes_libro <- c(
  "ggplot2", "readr", "dplyr", "tidyr", "stringr", "data.table",
  "class", "rpart", "randomForest", "ranger", "e1071", "naivebayes",
  "neuralnet", "cluster", "caret", "factoextra", "scales", "plotly", "DT"
)
faltantes <- paquetes_libro[!vapply(paquetes_libro, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltantes)) install.packages(faltantes)

dir.create("datos/covid19/procesados", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/muestras", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/diccionarios", showWarnings = FALSE, recursive = TRUE)

archivos_colab <- c(
  "util_graficas.R" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/util_graficas.R",
  "datos/atus_ml_preparado.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/atus_ml_preparado.csv",
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  "datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz",
  "datos/covid19/diccionarios/diccionario_covid19_ml.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/diccionarios/diccionario_covid19_ml.csv"
)
for (destino in names(archivos_colab)) {
  if (!file.exists(destino)) download.file(archivos_colab[[destino]], destino, mode = "wb", quiet = TRUE)
}
stopifnot(all(file.exists(names(archivos_colab))))
source("util_graficas.R")
cat("Entorno autónomo listo. R:", R.version.string, "\n")


# Agrupamiento con k-means

La formulación matemática de **distancias, función objetivo, centroides y agrupamiento k-means** se desarrolla con mayor profundidad
en los capítulos 2 y 18 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos de aprendizaje

Al finalizar este capítulo podrás:

- distinguir aprendizaje supervisado y no supervisado;
- explicar intuitivamente el algoritmo k-means;
- calcular centroides y asignar observaciones a grupos;
- implementar k-means manualmente y con `kmeans()`;
- seleccionar el número de grupos mediante el método del codo y silhouette;
- interpretar centroides y perfiles de los grupos;
- reconocer las limitaciones y los errores frecuentes;
- aplicar el algoritmo a datos reales.

## Del aprendizaje supervisado al no supervisado

En los capítulos anteriores conocíamos la variable de respuesta. En el aprendizaje no supervisado no existe una etiqueta conocida que indique el grupo correcto. El objetivo es descubrir estructura en los datos.

El **agrupamiento** o *clustering* reúne observaciones similares y separa observaciones diferentes.

## Idea intuitiva de k-means

k-means busca dividir las observaciones en $k$ grupos. Cada grupo se representa mediante un **centroide**, que es el promedio de sus observaciones.

El algoritmo repite dos operaciones:

1. asignar cada observación al centroide más cercano;
2. recalcular los centroides.

El proceso termina cuando las asignaciones dejan de cambiar o la mejora es muy pequeña.

## Función objetivo

k-means minimiza la suma de cuadrados dentro de los grupos:

$$
WSS=\sum_{j=1}^{k}\sum_{\mathbf{x}_i\in C_j}\|\mathbf{x}_i-\boldsymbol{\mu}_j\|^2
$$

Aquí, $C_j$ es el grupo $j$ y $\boldsymbol{\mu}_j$ es su centroide.

## Ejemplo manual

Considera seis puntos:

| Punto | x | y |
|---|---:|---:|
| A | 1 | 1 |
| B | 1.5 | 2 |
| C | 3 | 4 |
| D | 5 | 7 |
| E | 3.5 | 5 |
| F | 4.5 | 5 |

Si fijamos $k=2$, elegimos dos centroides iniciales, asignamos cada punto al más cercano y recalculamos los promedios de ambos grupos. Repetimos hasta estabilizar.

## Implementación básica en R


In [ ]:
datos <- data.frame(
  x = c(1, 1.5, 3, 5, 3.5, 4.5),
  y = c(1, 2, 4, 7, 5, 5)
)

datos


In [ ]:
set.seed(123)
modelo_km <- kmeans(datos, centers = 2, nstart = 25)
modelo_km


In [ ]:
datos$cluster <- factor(modelo_km$cluster)
plot(
  datos$x, datos$y,
  col = datos$cluster,
  pch = 19,
  xlab = "x", ylab = "y",
  main = "Agrupamiento k-means"
)
points(modelo_km$centers, pch = 8, cex = 2, lwd = 2)


## ¿Por qué usar `nstart`?

k-means depende de los centroides iniciales. `nstart = 25` ejecuta el algoritmo varias veces y conserva la solución con menor variación interna.

## Escalamiento de variables

Si una variable está medida en miles y otra entre 0 y 10, la primera dominará las distancias. Por ello suele ser necesario estandarizar:

$$
z=\frac{x-\bar{x}}{s}
$$


In [ ]:
X <- scale(iris[, 1:4])
head(X)


## Aplicación con `iris`


In [ ]:
set.seed(2026)
km_iris <- kmeans(X, centers = 3, nstart = 50)

table(
  Cluster = km_iris$cluster,
  Especie = iris$Species
)


Los números de los clusters no tienen significado previo. El grupo 1 no es mejor ni peor que el grupo 2.

## Interpretación de centroides


In [ ]:
km_iris$centers


Cada fila representa un perfil promedio estandarizado. Valores positivos indican niveles superiores a la media; valores negativos, inferiores.

## Método del codo

El método del codo compara la suma de cuadrados interna para distintos valores de $k$.


In [ ]:
valores_k <- 1:10
wss <- numeric(length(valores_k))

for (i in seq_along(valores_k)) {
  set.seed(123)
  ajuste <- kmeans(X, centers = valores_k[i], nstart = 25)
  wss[i] <- ajuste$tot.withinss
}

plot(
  valores_k, wss,
  type = "b",
  xlab = "Número de clusters k",
  ylab = "Suma de cuadrados interna",
  main = "Método del codo"
)


Se busca un punto a partir del cual añadir grupos produce mejoras pequeñas.

## Coeficiente silhouette

Silhouette compara cohesión interna y separación externa. Sus valores se encuentran entre -1 y 1:

- cerca de 1: agrupamiento claro;
- cerca de 0: observación fronteriza;
- negativo: posible asignación incorrecta.


## Laboratorio interactivo k-means

Modifica el número de grupos y observa cómo cambian los centroides y la separación.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


El laboratorio interactivo de k-means está disponible en la versión web del capítulo. Permite modificar $k$, el tamaño de la muestra y la dispersión de los grupos.

## Ventajas

- sencillo e intuitivo;
- rápido en conjuntos medianos y grandes;
- fácil de visualizar;
- útil para segmentación y exploración;
- disponible en R base.

## Limitaciones

- requiere fijar $k$;
- depende de la inicialización;
- es sensible a escalas y valores atípicos;
- favorece grupos aproximadamente esféricos;
- no maneja bien clusters con densidades muy diferentes;
- solo trabaja naturalmente con variables numéricas.

## Errores frecuentes

1. No estandarizar variables.
2. Elegir $k$ únicamente porque coincide con una expectativa previa.
3. Ejecutar una sola inicialización.
4. Interpretar los números de cluster como categorías ordenadas.
5. Incluir identificadores.
6. Ignorar valores atípicos.
7. Confundir clusters con clases verdaderas.
8. Describir grupos sin revisar sus centroides.

## Aplicación con datos reales

En accidentes de tránsito pueden agruparse municipios, horarios o tipos de accidente usando variables como número de víctimas, frecuencia, hora, zona y severidad. El objetivo no es predecir una etiqueta, sino descubrir perfiles semejantes.

## Actividad guiada

1. Selecciona cuatro variables numéricas de un conjunto real.
2. Trata valores faltantes.
3. Estandariza los datos.
4. Evalúa valores de $k$ entre 2 y 10.
5. Usa el método del codo.
6. Calcula silhouette.
7. Entrena el modelo final con `nstart = 50`.
8. Interpreta los centroides.
9. Asigna nombres descriptivos a los perfiles.
10. Explica las limitaciones del análisis.

## Ejercicios

1. Implementa una iteración manual de k-means.
2. Compara resultados con y sin estandarización.
3. Cambia la semilla y analiza la estabilidad.
4. Agrega valores atípicos.
5. Compara $k=2$, $k=3$ y $k=5$.
6. Contrasta el método del codo y silhouette.
7. Aplica k-means a datos ambientales o de accidentes.

## Caso aplicado B: k-means con COVID-19

En esta segunda ruta aplicada usamos datos abiertos de COVID-19 México 2022 para descubrir **perfiles de observaciones semejantes**. A diferencia de los capítulos de clasificación, k-means es un método no supervisado: por ello la variable `MURIO` **no participa en la formación de los grupos**.

> **Uso académico:** los clusters representan patrones estadísticos dentro de esta muestra. No son categorías clínicas, diagnósticos ni niveles de riesgo médico.

### Preparar las variables para agrupamiento

Usamos variables numéricas relacionadas con edad y comorbilidades. Las variables binarias se mantienen como 0/1 y todas se estandarizan para evitar que una escala domine las distancias.


In [ ]:
ruta_covid <- "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz"

if (file.exists(ruta_covid)) {
  covid_km <- readr::read_csv(ruta_covid, show_col_types = FALSE) |>
    dplyr::select(
      MURIO, EDAD, NEUMONIA, DIABETES, HIPERTENSION,
      OBESIDAD, RENAL_CRONICA, NUM_COMORBILIDADES
    ) |>
    tidyr::drop_na()

  set.seed(2026)
  covid_km <- covid_km |>
    dplyr::sample_n(min(12000, nrow(covid_km)))

  variables_km <- c(
    "EDAD", "NEUMONIA", "DIABETES", "HIPERTENSION",
    "OBESIDAD", "RENAL_CRONICA", "NUM_COMORBILIDADES"
  )

  X_covid_km <- scale(covid_km[variables_km])
}


`MURIO` se conserva únicamente para una interpretación posterior. No se incluye en `X_covid_km`, de modo que los clusters se forman sin conocer la variable de defunción.

### Explorar el número de grupos con el método del codo


In [ ]:
if (exists("X_covid_km")) {
  valores_k_covid <- 1:8
  wss_covid <- numeric(length(valores_k_covid))

  for (i in seq_along(valores_k_covid)) {
    set.seed(2026)
    ajuste <- kmeans(
      X_covid_km,
      centers = valores_k_covid[i],
      nstart = 30,
      iter.max = 100
    )
    wss_covid[i] <- ajuste$tot.withinss
  }

  plot(
    valores_k_covid, wss_covid,
    type = "b", pch = 19,
    xlab = "Número de clusters k",
    ylab = "Suma de cuadrados interna",
    main = "COVID-19: método del codo"
  )
}


El método del codo no elige automáticamente un valor perfecto de \(k\); ayuda a identificar cuándo agregar más clusters produce mejoras cada vez menores.

### Comparar silhouette

Para que el cálculo de distancias sea ligero, se evalúa silhouette sobre una submuestra reproducible.


In [ ]:
if (exists("X_covid_km")) {
  set.seed(2026)
  n_sil <- min(2500, nrow(X_covid_km))
  idx_sil <- sample(seq_len(nrow(X_covid_km)), n_sil)
  X_sil <- X_covid_km[idx_sil, , drop = FALSE]
  dist_sil <- dist(X_sil)

  sil_promedio <- sapply(2:6, function(k_actual) {
    set.seed(2026)
    km_tmp <- kmeans(X_sil, centers = k_actual, nstart = 30)
    sil_tmp <- cluster::silhouette(km_tmp$cluster, dist_sil)
    mean(sil_tmp[, "sil_width"])
  })

  data.frame(
    k = 2:6,
    silhouette_promedio = sil_promedio
  )
}


### Ajustar una solución didáctica con tres clusters

Para mantener la interpretación sencilla usamos \(k=3\). En un análisis formal, este valor debería justificarse conjuntamente con el codo, silhouette, estabilidad e interpretabilidad.


In [ ]:
if (exists("X_covid_km")) {
  set.seed(2026)
  modelo_covid_km <- kmeans(
    X_covid_km,
    centers = 3,
    nstart = 50,
    iter.max = 100
  )

  covid_km$cluster <- factor(modelo_covid_km$cluster)
  table(covid_km$cluster)
}


### Interpretar los centroides


In [ ]:
if (exists("modelo_covid_km")) {
  centroides_covid <- as.data.frame(modelo_covid_km$centers)
  centroides_covid$cluster <- factor(seq_len(nrow(centroides_covid)))
  centroides_covid
}


Como los datos están estandarizados, un centroide positivo indica que el cluster presenta, en promedio, valores superiores a la media de la muestra para esa variable; un valor negativo indica valores inferiores.

### Perfil descriptivo en unidades originales


In [ ]:
if (exists("modelo_covid_km")) {
  perfiles_covid <- covid_km |>
    dplyr::group_by(cluster) |>
    dplyr::summarise(
      n = dplyr::n(),
      edad_media = mean(EDAD),
      neumonia_pct = mean(NEUMONIA) * 100,
      diabetes_pct = mean(DIABETES) * 100,
      hipertension_pct = mean(HIPERTENSION) * 100,
      obesidad_pct = mean(OBESIDAD) * 100,
      renal_cronica_pct = mean(RENAL_CRONICA) * 100,
      comorbilidades_media = mean(NUM_COMORBILIDADES),
      .groups = "drop"
    )

  perfiles_covid
}


### Observar `MURIO` después de formar los grupos

Ahora sí usamos la variable `MURIO`, pero únicamente como una **descripción externa** de los clusters ya construidos.


In [ ]:
if (exists("modelo_covid_km")) {
  mortalidad_por_cluster <- covid_km |>
    dplyr::group_by(cluster) |>
    dplyr::summarise(
      casos = dplyr::n(),
      defunciones_registradas = sum(MURIO == 1),
      porcentaje_defuncion = mean(MURIO == 1) * 100,
      .groups = "drop"
    )

  mortalidad_por_cluster
}


Una diferencia en el porcentaje de defunción entre clusters **no demuestra causalidad** y tampoco convierte k-means en un modelo predictivo. Los grupos fueron definidos únicamente por semejanza en las variables de entrada.

### Visualización bidimensional mediante PCA

La siguiente gráfica usa PCA solo para proyectar los clusters en dos dimensiones y facilitar su visualización; el agrupamiento continúa siendo el realizado en el espacio estandarizado original.


In [ ]:
if (exists("modelo_covid_km")) {
  pca_covid_km <- prcomp(X_covid_km, center = FALSE, scale. = FALSE)

  grafica_covid_km <- data.frame(
    PC1 = pca_covid_km$x[, 1],
    PC2 = pca_covid_km$x[, 2],
    cluster = covid_km$cluster
  )

  ggplot2::ggplot(
    grafica_covid_km,
    ggplot2::aes(x = PC1, y = PC2, color = cluster)
  ) +
    ggplot2::geom_point(alpha = 0.35, size = 1.2) +
    ggplot2::labs(
      title = "Perfiles COVID-19 obtenidos con k-means",
      subtitle = "Proyección PCA para visualizar los clusters",
      x = "Componente principal 1",
      y = "Componente principal 2",
      color = "Cluster"
    ) +
    ggplot2::theme_minimal()
}


Este ejercicio muestra una diferencia fundamental: en clasificación usamos una etiqueta para aprender a predecirla; en clustering buscamos estructura sin etiqueta. Después podemos relacionar los grupos con variables externas para describirlos, pero no para afirmar que esas variables causaron los clusters.

## Laboratorio interactivo COVID: agrupamiento k-means

Este laboratorio utiliza una **submuestra reproducible de la muestra COVID-19 México 2022 incluida en el libro**. Puedes modificar el número de clusters y observar cómo cambian los perfiles en una proyección de componentes principales.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio interactivo COVID disponible en la versión web

La versión web permite cambiar el número de clusters y visualizar una submuestra COVID-19 2022 agrupada mediante k-means y proyectada con PCA.

## Materiales complementarios del capítulo

### Video del capítulo

<!-- colab-capitulo -->

Este capítulo cuenta con un **cuaderno autónomo de Google Colab**. Puede abrirse y ejecutarse de manera independiente, sin necesidad de ejecutar los capítulos anteriores.

[**Abrir este capítulo en Google Colab**](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/13-kmeans.ipynb)

::: <!-- /colab-capitulo -->

### Video del capítulo

Disponible en YouTube:

<https://youtu.be/mZgUlnSxrgA>

| Recurso | Descripción | Abrir o descargar |
|---|---|---|
| Presentación en PDF | Síntesis del capítulo para lectura o exposición. | [Abrir PDF](recursos/capitulo-13/capitulo-13-kmeans-presentacion.pdf) |
| Presentación editable | Diapositivas en PowerPoint. | [Descargar PPTX](recursos/capitulo-13/capitulo-13-kmeans-presentacion.pptx) |
| Infografía | Resumen visual del capítulo. | [Abrir infografía](recursos/capitulo-13/capitulo-13-kmeans-infografia.png) |

![Infografía del capítulo 13](recursos/capitulo-13/capitulo-13-kmeans-infografia.png)

Los materiales fueron creados con apoyo de NotebookLM de Google a partir del
contenido del libro y revisados y adaptados por el autor.

## Conclusión

k-means descubre grupos mediante distancias y centroides. Su utilidad depende de una preparación adecuada, una selección razonada de $k$ y una interpretación basada en los perfiles de cada cluster.

Con este capítulo cerramos la ruta principal de algoritmos del libro. Los apéndices reúnen el glosario, las referencias y los índices para consulta.
